In [ ]:
from pathlib import Path
from datasets import load_dataset
from transformers import (XLMRobertaTokenizerFast, XLMRobertaForMaskedLM,
                          DataCollatorForLanguageModeling, TrainingArguments, Trainer)
import hashlib, json

BASE = 'xlm-roberta-base'
CORPUS = Path('AsoSoft-large.txt')
CHECKPOINT = Path('sorani-xlmr-pretrained-checkpoint-15-epoch')
if not CORPUS.is_file():
    raise FileNotFoundError('Original pretraining corpus required: AsoSoft-large.txt')
if CHECKPOINT.exists():
    raise FileExistsError(f'Will not overwrite an existing checkpoint: {CHECKPOINT}')

tokenizer = XLMRobertaTokenizerFast.from_pretrained(BASE)
corpus = load_dataset('text', data_files={'train': str(CORPUS)})['train']
corpus = corpus.map(lambda batch: tokenizer(batch['text'], max_length=512, truncation=True),
                    batched=True, num_proc=4, remove_columns=['text'])
collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=True, mlm_probability=0.15)
model = XLMRobertaForMaskedLM.from_pretrained(BASE)
model.gradient_checkpointing_enable()


In [ ]:
args = TrainingArguments(
    output_dir='sorani_pretraining_runs',
    eval_strategy='no',
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    num_train_epochs=15,
    save_strategy='steps',
    save_steps=10000,
    save_total_limit=2,
    logging_steps=500,
    fp16=True,
)
trainer = Trainer(model=model, args=args, train_dataset=corpus, data_collator=collator)
trainer.train()
model.save_pretrained(CHECKPOINT, safe_serialization=True)
tokenizer.save_pretrained(CHECKPOINT)


In [ ]:
from pandas import DataFrame

DataFrame(trainer.state.log_history).to_csv(CHECKPOINT/'training_log.csv', index=False)
with CORPUS.open('rb') as f:
    h = hashlib.sha256()
    for chunk in iter(lambda: f.read(8*1024*1024), b''):
        h.update(chunk)
(CHECKPOINT/'training_record.json').write_text(json.dumps({
    'base_model':BASE, 'mlm_probability':0.15,
    'epochs_target':15, 'epochs_completed':trainer.state.epoch,
    'global_steps':trainer.state.global_step,
    'corpus_sha256':h.hexdigest()
},indent=2),encoding='utf-8')
print('Checkpoint:', CHECKPOINT.resolve(), 'Completed epochs:', trainer.state.epoch)
